In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found. Check your .env file.")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=api_key,
    temperature=0
)

print("Groq model initialized successfully.")

Groq model initialized successfully.


In [2]:
orders = {
    "ORD-4521": {
        "status": "Shipped",
        "location": "Pune Distribution Center",
        "estimated_delivery": "2026-10-25"
    },
    "ORD-4522": {
        "status": "Processing",
        "location": "Seller Warehouse",
        "estimated_delivery": "2026-10-28"
    },
    "ORD-4523": {
        "status": "Delivered",
        "location": "Customer Address",
        "estimated_delivery": "2026-10-20"
    }
}

print("Sample order records created.")
print(f"Total sample orders: {len(orders)}")

Sample order records created.
Total sample orders: 3


In [3]:
@tool
def lookup_order_status(order_id: str) -> str:
    """Look up an order's shipping status, current location, and estimated delivery using its order ID."""

    order_id = order_id.strip().upper()
    order = orders.get(order_id)

    if order is None:
        return f"No order found for order ID {order_id}."

    return (
        f"Order ID: {order_id}\n"
        f"Status: {order['status']}\n"
        f"Current location: {order['location']}\n"
        f"Estimated delivery: {order['estimated_delivery']}"
    )


print("Order lookup tool created successfully.")

# Test the tool manually
test_result = lookup_order_status.invoke({
    "order_id": "ORD-4521"
})

print(test_result)

Order lookup tool created successfully.
Order ID: ORD-4521
Status: Shipped
Current location: Pune Distribution Center
Estimated delivery: 2026-10-25


In [4]:
tools = [lookup_order_status]

llm_with_tools = llm.bind_tools(tools)

user_query = "Where is my order ORD-4521?"

messages = [
    HumanMessage(content=user_query)
]

response = llm_with_tools.invoke(messages)

print("User query:")
print(user_query)

print("\nModel-suggested tool calls:")
print(response.tool_calls)

if not response.tool_calls:
    raise RuntimeError(
        "The model did not suggest a tool call. Run this cell again."
    )
    

User query:
Where is my order ORD-4521?

Model-suggested tool calls:
[{'name': 'lookup_order_status', 'args': {'order_id': 'ORD-4521'}, 'id': 'fc_b4f83b3e-9d85-41cb-9e55-602b5741a697', 'type': 'tool_call'}]


In [5]:
tool_map = {
    "lookup_order_status": lookup_order_status
}

# Add the assistant response containing tool calls to the conversation
messages.append(response)

tool_results = []

for tool_call in response.tool_calls:
    tool_name = tool_call["name"]
    tool_args = tool_call["args"]

    if tool_name not in tool_map:
        raise ValueError(f"Unknown tool requested: {tool_name}")

    # Execute the suggested tool call manually
    result = tool_map[tool_name].invoke(tool_args)

    tool_results.append(result)

    print(f"\nTool name: {tool_name}")
    print("Tool arguments:", tool_args)
    print("Executed tool result:")
    print(result)

    # Return the tool result to the model's conversation
    messages.append(
        ToolMessage(
            content=str(result),
            tool_call_id=tool_call["id"]
        )
    )

print("\nAll suggested tool calls executed.")


Tool name: lookup_order_status
Tool arguments: {'order_id': 'ORD-4521'}
Executed tool result:
Order ID: ORD-4521
Status: Shipped
Current location: Pune Distribution Center
Estimated delivery: 2026-10-25

All suggested tool calls executed.


In [6]:
final_response = llm.invoke(messages)

print("Final answer from the model:")
print(final_response.content)

Final answer from the model:
Your order **ORD‑4521** has been shipped and is currently at our **Pune Distribution Center**.  

**Estimated delivery date:** **October 25, 2026**.

If you’d like any additional details (tracking link, changes to the delivery address, etc.) just let me know!


In [7]:
assert response.tool_calls, "No tool calls were suggested."

assert any(
    call["name"] == "lookup_order_status"
    for call in response.tool_calls
), "The order lookup tool was not called."

assert len(tool_results) > 0, "No tool results were generated."

assert any(
    "ORD-4521" in result and "Shipped" in result
    for result in tool_results
), "Expected order information was not returned."

assert final_response.content.strip(), "The final answer is empty."

print("Validation passed.")
print("The model suggested the order lookup tool.")
print("The tool was executed manually.")
print("The tool result was returned to the model.")
print("The final natural-language answer was generated.")

Validation passed.
The model suggested the order lookup tool.
The tool was executed manually.
The tool result was returned to the model.
The final natural-language answer was generated.


In [8]:
unknown_result = lookup_order_status.invoke({
    "order_id": "ORD-9999"
})

print(unknown_result)

assert "No order found" in unknown_result

print("Unknown order test passed.")

No order found for order ID ORD-9999.
Unknown order test passed.


In [9]:
print("Lab 6 - Exercise 1 completed successfully.")
print("Tool: lookup_order_status")
print("Method: @tool")
print("Model: Groq openai/gpt-oss-120b")
print("Workflow: Bind tool -> Suggest call -> Execute tool")
print("-> Return result -> Generate final answer")

Lab 6 - Exercise 1 completed successfully.
Tool: lookup_order_status
Method: @tool
Model: Groq openai/gpt-oss-120b
Workflow: Bind tool -> Suggest call -> Execute tool
-> Return result -> Generate final answer
